# Trader Execution

Walk-forward is the test design: train on the past, test on the next unseen stretch, with an
hard fenced split so nothing leaks across. This chapter's notebook runs the machine-learning 
workflow on Binance's 1-hour full-market data for all active USDT spot pairs (n=433) trading 
with three pinned currencies. This point-in-time screening method was adopted as recommended 
data standards for building cryptocurrency training data and machine learning pipelines. 

This was planned across five stages: i) building the labelled 1h dataset, ii) running variable 
selection tests to confirm best subset of indicator features, iii) fitting and training models 
head-to-head between logistic regression, random forest, LightGBM meeting a 60/40 confidence filter, 
iv) testing models and tuning hyperparameters on blind data to nominate `best model`.

All model performance metrics are compiled in the `outputs/AA-evals/` folder, comparing features and 
the ATR-scaled label from `inputs/build_dataset_1h.py`, the final-year split from
`inputs/train_model_1h.py`, the estimator list and per-model scoring from `inputs/train_model.py`,
and the four-metric evaluation from `inputs/eval_report.py`. Run it from
the project `.venv`, which holds `pandas-ta` and `TA-Lib` lirbaries. 

### Environment

In [1]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt

# Run this from the project .venv (MacPorts python 3.12), which holds pandas-ta and TA-Lib.
if sys.version_info[:2] < (3, 11):
    raise RuntimeError(f"Use the project .venv (Python 3.12); kernel is {sys.version.split()[0]}.")

# Shared modeling + evaluation code: the SAME functions the scripts run.
for cand in ["inputs", os.path.join("..", "inputs")]:
    if os.path.isdir(cand):
        sys.path.insert(0, os.path.abspath(cand)); break
import build_dataset_1h as bd        # 1h features, ATR-scaled label, point-in-time screen
import train_model as tm             # shared: build_models, evaluate, confidence_filtered, costs
import train_model_1h as t1          # 1h load + final-year split
import eval_report
HAVE_LGBM = tm.HAVE_LGBM

def _layer(mod):
    try:
        __import__(mod); return "yes"
    except Exception:
        return "no"

OUTPUTS = Path("outputs"); MODEL_DIR = OUTPUTS / "3B-model-training"; MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"env ready  .  python {sys.version.split()[0]}  .  lightgbm {'yes' if HAVE_LGBM else 'no'}"
      f"  .  pandas-ta {_layer('pandas_ta')}  .  TA-Lib {_layer('talib')}")

ModuleNotFoundError: No module named 'joblib'

## Training-Test Data

One row per coin per **hour**, over the full active USDT spot market, point-in-time screened so
each row would have passed the four-gate screen as of its own bar. Features are scale-invariant;
the label is an **ATR-scaled triple barrier** on a short day-trade horizon (default +2 ATR before
-1 ATR within 48 bars). The split follows the data standard: hold out the final ~1 year
out-of-sample, train on **all** prior history, with an embargo of one label horizon at the cut so
no label peeks across. A full rolling walk-forward is the planned upgrade. See `tasks/data-standards.md`.

In [2]:
# Knobs live in the scripts so the notebook cannot drift. Read-only here.
CONFIG = dict(
    oos_days     = t1.OOS_DAYS,         # final year held out of sample
    embargo_days = t1.EMBARGO_DAYS,     # = label horizon in days
    conf_hi      = tm.CONF_HI,          # confidence filter act-long (Keller Metric 1)
    conf_lo      = tm.CONF_LO,          # act-short / stand-aside
    cost_pct     = tm.COST_PCT,         # round-trip drag (fee + slippage), Metric 2
)
lab = bd.LABEL
print(f"label: +{lab['tgt_atr']} ATR before -{lab['stp_atr']} ATR within {lab['horizon_bars']} bars "
      f"({lab['horizon_bars']//bd.BARS_PER_DAY}d)  .  hold out final {CONFIG['oos_days']}d, "
      f"embargo +/-{CONFIG['embargo_days']}d  .  confidence {CONFIG['conf_lo']}-{CONFIG['conf_hi']}  .  "
      f"cost {CONFIG['cost_pct']:.2f}% round trip")

NameError: name 't1' is not defined

### Import Data

Loads the 1h dataset at `inputs/binance-data/dataset_1h_allmarket.csv` (via `bd.DATASET_PATH`),
built offline from the `data.binance.vision` archives by `inputs/build_dataset_1h.py`. `t1.load`
reads it and keeps only the point-in-time `in_sample` rows. If it is not built yet (the overnight
download/build is still running), the cell says how to build it.

In [3]:
ds = bd.DATASET_PATH            # inputs/binance-data/dataset_1h_allmarket.csv
if os.path.exists(ds):
    df = t1.load(ds)            # reads 'datetime', keeps in_sample rows
    feat = bd.feature_columns(df)
    print(f"loaded {ds}\n{len(df):,} in-sample rows  .  {len(feat)} features  .  "
          f"{df['datetime'].min()} to {df['datetime'].max()}  .  base rate {df['label'].mean():.3f}")
else:
    df = None; feat = []
    print("1h dataset not built yet. Build it from the project .venv:\n"
          "  .venv/bin/python inputs/build_dataset_1h.py\n"
          "(or wait for the overnight auto-eval, tasks/run_auto_eval.sh, to produce it).")

NameError: name 'bd' is not defined

### Model Features Collection

The candidate set is broad by design - tomorrow's variable-selection pass (likelihood-ratio tests
and elastic-net regularization paths) prunes it. It spans two window families (a wall-clock family
= the daily windows x24, and a shorter intraday family), an in-house extra-indicator block
(Williams %R, Stochastic, CCI, CMF, MFI, ADX/DMI, Aroon), the trade-flow imbalance, and - when run
from the `.venv` - optional pandas-ta (PPO, TRIX, Vortex, CMO, Fisher, Chande Kroll Stop) and
TA-Lib (SAR, MAMA, Ultimate Oscillator, Hilbert cycle features, candlestick patterns) layers. All
causal and scale-invariant.

In [4]:
if df is not None:
    fam = [("wall-clock (wc)", "f_wc_"), ("intraday (hr)", "f_hr_"),
           ("in-house TA", "f_ta_"), ("pandas-ta", "f_ta_pta_"),
           ("TA-Lib", "f_tl_"), ("flow", "f_flow_")]
    for name, pre in fam:
        if pre == "f_ta_":
            cols = [c for c in feat if c.startswith("f_ta_") and not c.startswith("f_ta_pta_")]
        else:
            cols = [c for c in feat if c.startswith(pre)]
        print(f"  {name:16s}: {len(cols)}")
    print(f"  {'TOTAL':16s}: {len(feat)}")
    display(df[feat].describe().T[["mean", "std", "min", "max"]].round(3))

NameError: name 'df' is not defined

## Variable Selection

Stage ii: prune the broad candidate set to the subset that earns its place, BEFORE the head-to-head
and before any hyperparameter tuning, using the TRAINING window only so the final-year hold-out
stays untouched (otherwise the blind score is no longer blind).

- **Elastic-net path (the glmnet analogue).** A regularized logistic regression (`saga`,
  `penalty="elasticnet"`) over the standardized features on the training split; the features with
  non-zero coefficients at the chosen penalty are the selected subset. Sweep `l1_ratio` and `C`
  (the lambda / mixing knobs) to trace the path.
- **Likelihood-ratio test.** For the logistic model, compare nested fits (with vs without a feature
  or block) by the chi-square of the deviance difference. The tree models (RF, LightGBM) lack that
  nested-likelihood structure, so use permutation importance for them instead.

The selected subset feeds the models below. A Brier score (RMSE on predicted probabilities, the
`caret`-style classification RMSE) is printed as a calibration check.

In [1]:
# Stage ii: elastic-net variable selection on the TRAINING split only (the blind year is untouched).
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss

if df is not None and feat:
    tr_vs, te_vs, _cut = t1.split(df)                 # same final-year split; select on TRAIN
    Xtr_vs = tr_vs[feat].astype(float).fillna(0.0); ytr_vs = tr_vs["label"]
    sc = StandardScaler().fit(Xtr_vs)
    enet = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, C=0.1,
                              max_iter=5000, class_weight="balanced")
    enet.fit(sc.transform(Xtr_vs), ytr_vs)
    coef = pd.Series(enet.coef_[0], index=feat)
    SELECTED_FEATURES = list(coef[coef.abs() > 1e-6].abs().sort_values(ascending=False).index)
    print(f"elastic-net kept {len(SELECTED_FEATURES)} of {len(feat)} features (l1_ratio=0.5, C=0.1).")
    print("top kept:", ", ".join(SELECTED_FEATURES[:15]))
    p_tr = enet.predict_proba(sc.transform(Xtr_vs))[:, 1]
    print(f"in-sample Brier (lower = better calibrated): {brier_score_loss(ytr_vs, p_tr):.4f}")
    print("\nTo train the head-to-head on this subset, set  feat = SELECTED_FEATURES  then re-run the "
          "training cells. Sweep l1_ratio / C for the full path; add a per-block LRT as needed.")
else:
    SELECTED_FEATURES = []
    print("build the 1h dataset first (see Import Data).")

ModuleNotFoundError: No module named 'sklearn'

## Model Training

Train on all history before the final-year cut, score once on the held-out final year. Three
models compete: logistic regression, random forest, LightGBM (Tier 1). Each is reported at the
0.5 threshold and under the 60/40 confidence filter (Keller Metric 1); read precision against the
base rate, well below 0.5. The full record - Metric 1, Metric 2 (P&L after the 0.20% cost), and
Metric 3 (AUC by volatility regime) - is written to `outputs/AA-evals/` by
`eval_report.write_comparison`, tagged evaluation type "head-to-head (1h)", so it sits beside the
other runs in `evaluation-scores.md`.

In [2]:
# Split and scoring come from the scripts: t1.split for the 1h final-year hold-out; tm.evaluate,
# tm.build_models, tm.confidence_filtered shared with inputs/train_model.py. Nothing reimplemented.
train, test, cut = t1.split(df)
Xtr, ytr, Xte, yte = train[feat], train["label"], test[feat], test["label"]
base_te = yte.mean()
span = lambda d: f"{d['datetime'].min().date()} to {d['datetime'].max().date()}"
print(f"train {len(train):,} ({span(train)})  .  test {len(test):,} ({span(test)})  .  "
      f"cut {pd.Timestamp(cut).date()}  embargo +/-{t1.EMBARGO_DAYS}d  .  base rate {base_te:.3f}")

NameError: name 't1' is not defined

In [3]:
lines, scored = [], []
for name, mdl in tm.build_models(HAVE_LGBM):
    prec, m = tm.evaluate(name, mdl, Xtr, ytr, Xte, yte, base_te, lines)
    scored.append((prec, name, mdl, m))
print("\n".join(lines))
_, best_name, best_model, best = max(scored, key=lambda t: t[0])
print(f"\nchosen: {best_name}  (precision {best['prec']:.3f} vs base {base_te:.3f}, AUC {best['auc']:.3f})")

NameError: name 'tm' is not defined

In [4]:
# Honesty gate: precision must clearly beat the base rate and AUC clear 0.55. It does not
# trade; Metric 2 (next cell) is the money test.
margin = best["prec"] - base_te
go = (best["prec"] > base_te + 0.05) and (best["auc"] > 0.55)
verdict = "GO" if go else "NO-GO"
print("HONESTY GATE:", "GO (edge survives out-of-sample)" if go
      else "NO-GO (no demonstrable edge - do not trade)")
joblib.dump({"model": best_model, "features": feat, "name": best_name,
             "trained_through": str(pd.Timestamp(cut).date()),
             "test_base_rate": float(base_te), "go": bool(go)}, MODEL_DIR / "model_1h.joblib")
summary = (f"{best_name}: test precision(buy)={best['prec']:.3f} base_rate={base_te:.3f} "
           f"lift={margin:+.3f} AUC={best['auc']:.3f} recall={best['rec']:.3f} acc={best['acc']:.3f} -> {verdict}")
(MODEL_DIR / "model_metrics_1h.txt").write_text(summary + "\n\n" + "\n".join(lines) + "\n")
print("saved", MODEL_DIR / "model_1h.joblib", "and model_metrics_1h.txt")

NameError: name 'best' is not defined

In [5]:
# Metrics 2 (P&L after costs) and 3 (regime-stratified AUC) plus AA-evals bookkeeping, from
# inputs/eval_report.py - the same record the script writes. Appends one row to
# outputs/AA-evals/evaluation-scores.md (+ .pdf, .docx), tagged "head-to-head (1h)".
imp = getattr(best_model, "feature_importances_", None)
regime = "f_wc_rv_long" if "f_wc_rv_long" in test.columns else next((c for c in feat if "rv_long" in c), None)
meta = dict(dataset_rows=len(df), n_features=len(feat),
            train_rows=len(train), test_rows=len(test), base_rate=float(base_te),
            cut=str(pd.Timestamp(cut).date()), embargo=t1.EMBARGO_DAYS,
            conf_hi=tm.CONF_HI, conf_lo=tm.CONF_LO, chosen=best_name, verdict=verdict,
            eval_type="head-to-head (1h)",
            dataset_label=f"{len(df):,}r / {len(feat)}f (1h all-market)",
            fi_names=list(feat) if imp is not None else None,
            fi_values=[float(v) for v in imp] if imp is not None else None,
            regime_vol=test[regime].tolist() if regime else None,
            trade_ret=test["trade_ret"].tolist() if "trade_ret" in test.columns else None,
            cost_pct=tm.COST_PCT)
results = [m for (_, _, _, m) in scored]
rec = eval_report.write_comparison(str(OUTPUTS / "AA-evals"), results, yte, meta)
print("evaluation record:", rec["md"])

NameError: name 'best_model' is not defined

In [6]:
# Teaching display: the strongest tree model's top features. The AA-evals record above
# already saves this chart; this is just an inline look.
tree = next((mdl for (_, name, mdl, _) in scored
             if name in ("LightGBM", "RandomForest")), None)
imp = getattr(tree, "feature_importances_", None) if tree is not None else None
if imp is not None:
    order = np.argsort(imp)[::-1][:15]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh([feat[i] for i in order][::-1], imp[order][::-1], color="#0B3D66")
    ax.set_title(f"{tree.__class__.__name__}: top 15 feature importances")
    fig.tight_layout()
    (OUTPUTS / "PNG").mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUTS / "PNG" / "3B-feature-importance.png", dpi=160)
    plt.show()

## Model Tuning

Two sweeps, both scored on the after-fee out-of-sample scoreboard and logged to the consolidated
`outputs/AA-evals/evaluation-scores.md`:

- **Priority 1b - label geometry** (`inputs/sweep_label_1h.py`): the ATR-scaled triple barrier
  swept over (target, stop, horizon). Each cell reports its own base rate, the breakeven win rate
  stop/(stop+target), and net P&L/trade after cost, so a lopsided geometry is obvious.
- **Priority 1a - exit geometry** (`inputs/walkforward.py`): stop and take-profit, plus per-coin
  trailing stops and a time-decaying take-profit - the same question from the exit side.

Settle the two together, then unify the forked label/exit configs. The broad feature set is left
broad here on purpose; the variable-selection pass (likelihood-ratio tests, elastic-net paths)
prunes predictors before final fitting.

In [7]:
# The sweeps are full-market and run as scripts from the project .venv:
#   .venv/bin/python inputs/sweep_label_1h.py    # Priority 1b: label geometry (target, stop, horizon)
#   .venv/bin/python inputs/walkforward.py       # Priority 1a: exit geometry (stop, take-profit, trail)
# Both append to the consolidated scoreboard; here we render that one table so tuning reads off it.
from IPython.display import Markdown, display
scoreboard = OUTPUTS / "AA-evals" / "evaluation-scores.md"
if scoreboard.exists():
    display(Markdown(scoreboard.read_text()))
else:
    print("no evaluation-scores.md yet - run a model (cells above) or a sweep script first.")
# Quick inline 1b probe (small grid; the full grid is the script above):
#   import sweep_label_1h as sw
#   coins, scols = sw.precompute(bd.DEFAULT_KLINES_ROOT, bd.DEFAULT_FLOW_CSV, None)
#   sw.score_cell(coins, scols, 3.0, 1.0, 48, tm.COST_PCT/100.0, tm.CONF_HI)

NameError: name 'OUTPUTS' is not defined

## Stability

Confirm any edge is not an artifact: parameter stability, results split by market type,
coin-flip and buy-and-hold baselines, an optional bootstrap on trade returns. Only then
paper trade, then a tiny live allocation. No live trading until a configuration clearly
beats buy-and-hold and a coin flip, out-of-sample and after fees.